Q2.) TASK 0)

In [ ]:
import torch
import torch.nn as nn
import random

# Load names
# Read the file containing generated Indian names.
# Each line corresponds to one name.
# We convert all names to lowercase for consistency and to reduce vocabulary size.

with open("TrainingNames.txt") as f:
    names = [line.strip().lower() for line in f]

# Create vocab
# We create a set of all unique characters present in the dataset.
# This is a character-level model, so each character is treated as a token.
# We also add special tokens:
#   <s>  : Start of sequence
#   </s> : End of sequence
# These help the model learn when a name begins and ends.
chars = sorted(list(set("".join(names)))) + ['<s>', '</s>']
# Create mappings:
# stoi (string → index): maps each character to a unique integer ID
# itos (index → string): reverse mapping for decoding
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}

vocab_size = len(stoi)

# Converts a name (string) into a sequence of integer indices.
# Example:
#   "ram" → [<s>, r, a, m, </s>]
# This is required because neural networks work with numbers, not text
def encode(name):
    return [stoi['<s>']] + [stoi[c] for c in name] + [stoi['</s>']]
# Converts a sequence of indices back into a readable string.
# Useful for generating names from the model output.

def decode(indices):
    return "".join([itos[i] for i in indices])
#Apply encoding to all names in the dataset.
# Now 'data' is a list of sequences of integers, ready for model training.
data = [encode(n) for n in names]

/home/suraj/.vscode-server/cuda/heysk/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Q2.) TASK 1)

In [ ]:

# Vanilla RNN
#  Recurrent Neural Network for character-level sequence modeling. The goal is to predict the next character
# given previous characters in a name.
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        # EMBEDDING LAYER
        
        # Converts each character index into a dense vector representation.
        # Instead of using one-hot vectors (sparse), embeddings allow the model
        # to learn meaningful relationships between characters.
        # Input shape: (batch_size, sequence_length)
        # Output shape: (batch_size, sequence_length, embed_size)
        self.embed = nn.Embedding(vocab_size, embed_size)
        # RNN PARAMETERS (MANUALLY DEFINED)
      
        # Wx: Weight matrix for input → hidden transformation
        # Wh: Weight matrix for hidden → hidden (temporal dependency)
        # b : Bias for hidden state update
        #
        # These define the recurrence relation:
        #   h_t = tanh(x_t * Wx + h_{t-1} * Wh + b)
        #
        # We initialize with small random values to stabilize training.
        self.Wx = nn.Parameter(torch.randn(embed_size, hidden_size)*0.1)
        self.Wh = nn.Parameter(torch.randn(hidden_size, hidden_size)*0.1)
        self.b = nn.Parameter(torch.zeros(hidden_size))
        # OUTPUT LAYER PARAMETERS
       
        # Wo: Hidden → output transformation
        # bo: Output bias
        #
        # The output at each time step is a vector of size vocab_size,
        # representing logits (unnormalized probabilities) for each character.
        self.Wo = nn.Parameter(torch.randn(hidden_size, vocab_size)*0.1)
        self.bo = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x):
        # FORWARD PASS
      
        # x: Input tensor of shape (batch_size, sequence_length)
        # Each element is a character index.

        # Convert indices to embeddings
        x = self.embed(x)
        # Extract batch size (B) and sequence length (T)
        B, T, _ = x.shape
        # INITIAL HIDDEN STATE
       
        # Initialize hidden state h_0 as zeros.
        # Shape: (batch_size, hidden_size)
        # This represents no prior context at the start of the sequence.
        h = torch.zeros(B, self.Wh.shape[0], device=x.device)
        outputs = []
        # TIME-STEP ITERATION (UNROLLED RNN)
     
        # We process the sequence one time step at a time.
        # This explicitly models temporal dependencies.
        for t in range(T):
            # HIDDEN STATE UPDATE
         
            # x[:, t] → embedding of current character (B, embed_size)
            # h       → previous hidden state (B, hidden_size)
            #
            # Compute new hidden state using tanh activation:
            #   h_t = tanh(x_t * Wx + h_{t-1} * Wh + b)
            #
            # This captures both:
            #   - current input information
            #   - past context (through h_{t-1})
            h = torch.tanh(x[:, t] @ self.Wx + h @ self.Wh + self.b)
            # OUTPUT COMPUTATION
          
            # Map hidden state to vocabulary space:
            #   output_t = h_t * Wo + bo
            #
            # Output shape: (B, vocab_size)
            # These are logits (before softmax).
            out = h @ self.Wo + self.bo
            outputs.append(out.unsqueeze(1))
# Combine outputs from all time steps into a single tensor:
        # Final shape: (B, T, vocab_size)
        return torch.cat(outputs, dim=1)

In [ ]:

# LSTM CELL
# This class implements a single LSTM cell manually (without using nn.LSTM).
# LSTM solves the vanishing gradient problem of vanilla RNN by introducing
# a memory cell and gating mechanisms that control information flow.

class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        # COMBINED WEIGHT MATRICES
        # -----------------------------
        # Instead of defining separate weights for each gate (i, f, g, o),
        # we combine them into a single matrix for efficiency.
        #
        # W: input → gates
        # U: hidden → gates
        # b: bias for all gates
        #
        # Total output size = 4 * hidden_size because we compute:
        #   input gate (i), forget gate (f), candidate (g), output gate (o)
        super().__init__()
        self.W = nn.Parameter(torch.randn(input_size, 4*hidden_size) * 0.1)
        self.U = nn.Parameter(torch.randn(hidden_size, 4*hidden_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(4*hidden_size))
        self.hidden_size = hidden_size

    def forward(self, x, h, c):
        # LSTM FORWARD STEP
        
        # x : current input (B, input_size)
        # h : previous hidden state (B, hidden_size)
        # c : previous cell state (B, hidden_size)

        # Compute all gate activations together for efficiency:
        # gates = xW + hU + b
        # Shape: (B, 4 * hidden_size)
        gates = x @ self.W + h @ self.U + self.b
        i, f, g, o = gates.chunk(4, dim=1)
        # SPLIT INTO 4 GATES
        # -----------------------------
        # Divide the combined tensor into:
        # i → input gate
        # f → forget gate
        # g → candidate (new memory content)
        # o → output gate
        # APPLY NON-LINEARITIES
        # -----------------------------
        # Sigmoid → values between 0 and 1 (acts like a "gate")
        # Tanh → values between -1 and 1 (content transformation)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        # CELL STATE UPDATE 

        # c_t = f * c_{t-1} + i * g
        #
        # - f * c → retains previous memory
        # - i * g → adds new information
        #
        # This linear path allows gradients to flow easily → solves vanishing gradient problem
        c = f * c + i * g
        # h_t = o * tanh(c_t)
        #
        # Hidden state is a filtered version of cell state
        # controlled by the output gate
        h = o * torch.tanh(c)

        return h, c


# BIDIRECTIONAL LSTM (BLSTM)
# This model processes the sequence in BOTH directions: Forward (left → right), Backward (right → left)

# This allows the model to capture: past context (prefix), future context (suffix)
# Very useful for sequence tasks like name generation.
class BLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        # EMBEDDING LAYER
      
        # Converts character indices into dense vectors
        self.embed = nn.Embedding(vocab_size, embed_size)
         # TWO LSTM DIRECTIONS
        # lstm_f → forward direction
        # lstm_b → backward direction
        self.lstm_f = LSTMCell(embed_size, hidden_size)
        self.lstm_b = LSTMCell(embed_size, hidden_size)
        # DROPOUT FOR REGULARIZATION
        # Helps reduce overfitting by randomly zeroing activations
        self.dropout = nn.Dropout(0.3)
        # OUTPUT LAYER
        # Since we concatenate forward + backward outputs,
        # input size = 2 * hidden_size
        self.Wo = nn.Parameter(torch.randn(hidden_size*2, vocab_size) * 0.1)
        self.bo = nn.Parameter(torch.zeros(vocab_size))
        self.hidden_size = hidden_size

    def forward(self, x):
        x = self.embed(x)
        B, T, _ = x.shape

        h_f = torch.zeros(B, self.hidden_size, device=x.device)
        c_f = torch.zeros_like(h_f)

        h_b = torch.zeros(B, self.hidden_size, device=x.device)
        c_b = torch.zeros_like(h_b)
       
        out_f, out_b = [], []
        # forward PASS (RIGHT → LEFT)
      
        for t in range(T):
            h_f, c_f = self.lstm_f(x[:, t], h_f, c_f)
            out_f.append(h_f.unsqueeze(1))
            # BACKWARD PASS (RIGHT → LEFT)
        for t in reversed(range(T)):
            h_b, c_b = self.lstm_b(x[:, t], h_b, c_b)
            out_b.insert(0, h_b.unsqueeze(1))
         # CONCATENATE BOTH DIRECTIONS

        # Shape:
        # forward  → (B, T, hidden_size)
        # backward → (B, T, hidden_size)
        # combined → (B, T, 2 * hidden_size)
        out = torch.cat([torch.cat(out_f, 1), torch.cat(out_b, 1)], dim=2)
        out = self.dropout(out)
            # Map combined representation to vocabulary space
        # Output shape: (B, T, vocab_size)
        return out @ self.Wo + self.bo


In [ ]:
# ATTENTION RNN 
class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        # Converts character indices to dense vectors
        self.embed = nn.Embedding(vocab_size, embed_size)

        # Basic RNN parameters (same idea as vanilla RNN)
        # Wx: input → hidden, Wh: hidden → hidden
        self.Wx = nn.Parameter(torch.randn(embed_size, hidden_size) * 0.1)
        self.Wh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.bh = nn.Parameter(torch.zeros(hidden_size))

        # Linear layers to create Query, Key, Value for attention
        # These help the model decide which past positions to focus on
        self.Wq = nn.Linear(hidden_size, hidden_size)
        self.Wk = nn.Linear(hidden_size, hidden_size)
        self.Wv = nn.Linear(hidden_size, hidden_size)

        # Dropout to reduce overfitting
        self.dropout = nn.Dropout(0.3)

        # Final layer to map to vocabulary size
        # We use both current hidden state + context vector
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x):
        # x shape: (batch, sequence_length)
        x = self.embed(x)  # (B, T, embed_size)
        B, T, H = x.shape

        # Initial hidden state (no past info at start)
        h = torch.zeros(B, H, device=x.device)

        states = []

        # Step 1: Build hidden states for entire sequence
        # This is like a normal RNN pass
        for t in range(T):
            h = torch.tanh(x[:, t] @ self.Wx + h @ self.Wh + self.bh)
            states.append(h.unsqueeze(1))

        # Combine all hidden states → (B, T, H)
        states = torch.cat(states, dim=1)

        # Step 2: Precompute Keys and Values for all time steps
        K = self.Wk(states)   # (B, T, H)
        V = self.Wv(states)   # (B, T, H)

        outputs = []

        # Step 3: Apply attention at each time step
        for t in range(T):
            ht = states[:, t]  # current hidden state (B, H)

            # Query comes from current state
            Q = self.Wq(ht).unsqueeze(1)  # (B, 1, H)

            # Causal mask: prevents looking at future positions
            # This ensures prediction only depends on past tokens
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1)
            mask = mask[t].unsqueeze(0).unsqueeze(1)  # (1,1,T)

            # Compute attention scores (Q · K^T)
            scores = torch.matmul(Q, K.transpose(1, 2)) / (H ** 0.5)

            # Apply mask so future positions are ignored
            scores = scores.masked_fill(mask.bool(), float('-inf'))

            # Convert scores to probabilities
            weights = torch.softmax(scores, dim=-1)

            # Compute context vector as weighted sum of values
            context = torch.matmul(weights, V).squeeze(1)  # (B, H)

            # Combine context with current hidden state
            combined = torch.cat([context, ht], dim=1)

            # Apply dropout for regularization
            combined = self.dropout(combined)

            # Final output (logits for next character)
            out = self.fc(combined)

            outputs.append(out.unsqueeze(1))

        # Return outputs for all time steps → (B, T, vocab_size)
        return torch.cat(outputs, dim=1)


A basic additive attention mechanism is used where attention scores are computed
for each hidden state using a learnable scoring function. The scores are normalized
using softmax to obtain attention weights. A context vector is computed as the
weighted sum of all hidden states, which is then used for prediction.

Q2.) Task 2

In [ ]:
import torch
import torch.nn as nn
import random


# Fix random seed so results remain same every run
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read names from file and convert to lowercase
with open("TrainingNames.txt") as f:
    names = [line.strip().lower() for line in f]

# Build character vocabulary from all names
# Also add start <s> and end </s> tokens
chars = sorted(list(set("".join(names)))) + ['<s>', '</s>']

# Mapping char → index and index → char
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

# Total number of unique characters
vocab_size = len(stoi)

# Convert name string to list of indices
def encode(name):
    return [stoi['<s>']] + [stoi[c] for c in name] + [stoi['</s>']]

# Encode all names for training
data = [encode(n) for n in names]

# VANILLA RNN

# Basic RNN without gates, processes sequence step by step
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        # Convert characters to embeddings
        self.embed = nn.Embedding(vocab_size, embed_size)

        # Input → hidden and hidden → hidden weights
        self.Wx = nn.Parameter(torch.randn(embed_size, hidden_size)*0.1)
        self.Wh = nn.Parameter(torch.randn(hidden_size, hidden_size)*0.1)
        self.b = nn.Parameter(torch.zeros(hidden_size))

        # Hidden → output layer
        self.Wo = nn.Parameter(torch.randn(hidden_size, vocab_size)*0.1)
        self.bo = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x):
        x = self.embed(x)  # (B,T,embed)
        B, T, _ = x.shape

        # Initial hidden state
        h = torch.zeros(B, self.Wh.shape[0], device=x.device)

        outputs = []

        # Process sequence step by step
        for t in range(T):
            h = torch.tanh(x[:, t] @ self.Wx + h @ self.Wh + self.b)
            out = h @ self.Wo + self.bo
            outputs.append(out.unsqueeze(1))

        # Combine all time step outputs
        return torch.cat(outputs, dim=1)

# LSTM CELL
# Handles long-term dependencies using gates
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        # Combined weights for all gates
        self.W = nn.Parameter(torch.randn(input_size, 4*hidden_size) * 0.1)
        self.U = nn.Parameter(torch.randn(hidden_size, 4*hidden_size) * 0.1)
        self.b = nn.Parameter(torch.zeros(4*hidden_size))

        self.hidden_size = hidden_size

    def forward(self, x, h, c):
        # Compute all gates together
        gates = x @ self.W + h @ self.U + self.b

        # Split into 4 parts
        i, f, g, o = gates.chunk(4, dim=1)

        # Apply activations
        i = torch.sigmoid(i)  # input gate
        f = torch.sigmoid(f)  # forget gate
        o = torch.sigmoid(o)  # output gate
        g = torch.tanh(g)     # candidate

        # Update cell and hidden state
        c = f * c + i * g
        h = o * torch.tanh(c)

        return h, c


# BIDIRECTIONAL LSTM
# Uses both forward and backward context
class BLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_size)

        # Two directions
        self.lstm_f = LSTMCell(embed_size, hidden_size)
        self.lstm_b = LSTMCell(embed_size, hidden_size)

        # Dropout to reduce overfitting
        self.dropout = nn.Dropout(0.3)

        # Output layer after combining both directions
        self.Wo = nn.Parameter(torch.randn(hidden_size*2, vocab_size) * 0.1)
        self.bo = nn.Parameter(torch.zeros(vocab_size))

        self.hidden_size = hidden_size

    def forward(self, x):
        x = self.embed(x)
        B, T, _ = x.shape

        # Initialize forward states
        h_f = torch.zeros(B, self.hidden_size, device=x.device)
        c_f = torch.zeros_like(h_f)

        # Initialize backward states
        h_b = torch.zeros(B, self.hidden_size, device=x.device)
        c_b = torch.zeros_like(h_b)

        out_f, out_b = [], []

        # Forward direction
        for t in range(T):
            h_f, c_f = self.lstm_f(x[:, t], h_f, c_f)
            out_f.append(h_f.unsqueeze(1))

        # Backward direction
        for t in reversed(range(T)):
            h_b, c_b = self.lstm_b(x[:, t], h_b, c_b)
            out_b.insert(0, h_b.unsqueeze(1))

        # Combine forward and backward outputs
        out = torch.cat([torch.cat(out_f, 1), torch.cat(out_b, 1)], dim=2)

        out = self.dropout(out)

        return out @ self.Wo + self.bo

# TRAIN FUNCTION
def train(model, data, epochs=10, lr=0.0005):

    # Adam optimizer with small weight decay
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-5
    )

    # Loss for next character prediction
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        
        model.train()
        total_loss = 0

        # Shuffle data each epoch
        random.shuffle(data)

        for seq in data:
            # Input sequence (except last)
            x = torch.tensor(seq[:-1], device=device).unsqueeze(0)

            # Target sequence (shifted)
            y = torch.tensor(seq[1:], device=device).unsqueeze(0)

            optimizer.zero_grad()

            out = model(x)

            # Compute loss
            loss = loss_fn(out.view(-1, out.size(-1)), y.view(-1))

            # Backpropagation
            loss.backward()

            # Prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

            optimizer.step()

            total_loss += loss.item() * x.size(1)

        print(f"Epoch {epoch+1}, Loss: {total_loss:.2f}")

# NOVELTY RATE
# Percentage of generated names not seen in training data
def compute_novelty(generated, training_set):
    training_set = set(training_set)
    novel = sum(1 for name in generated if name not in training_set)
    return novel / len(generated)

# DIVERSITY
# Fraction of unique generated names
def compute_diversity(generated):
    return len(set(generated)) / len(generated)

# NAME GENERATION
def generate_name(model, max_len=20):
    model.eval()

    # Start with <s> token
    idx = torch.tensor([[stoi['<s>']]], device=device)

    result = []

    with torch.no_grad():
        for _ in range(max_len):

            out = model(idx)
            logits = out[0, -1]

            # Temperature controls randomness
            probs = torch.softmax(logits / 1.2, dim=0)

            # Sample next character
            idx_next = torch.multinomial(probs, 1).item()

            char = itos[idx_next]

            # Stop if end token
            if char == '</s>':
                break

            # Ignore start token if generated again
            if char == '<s>':
                continue

            result.append(char)

            # Append new character to input
            idx = torch.cat(
                [idx, torch.tensor([[idx_next]], device=device)],
                dim=1
            )

    return "".join(result)

# Generate multiple names
def generate_samples(model, n=200):
    return [generate_name(model) for _ in range(n)]

# TRAIN MODELS
vanilla = VanillaRNN(vocab_size, 64, 64).to(device)
blstm = BLSTM(vocab_size, 64, 64).to(device)
attn = AttentionRNN(vocab_size, 64, 64).to(device)

print("Training Vanilla RNN...")
train(vanilla, data)

print("Training BLSTM...")
train(blstm, data)

print("\nTraining Attention...")
train(attn, data, epochs=15, lr=0.0003)

# EVALUATE
models = {"VanillaRNN":vanilla,"BLSTM": blstm, "Attention": attn}

for name, model in models.items():
    print(f"\n=== {name} ===")

    gen = generate_samples(model)

    print("Novelty:", compute_novelty(gen, names))
    print("Diversity:", compute_diversity(gen))
    print("Samples:", gen[:5])

Training Vanilla RNN...
Epoch 1, Loss: 34532.82
Epoch 2, Loss: 26996.07
Epoch 3, Loss: 24415.59
Epoch 4, Loss: 22740.41
Epoch 5, Loss: 21618.13
Epoch 6, Loss: 20730.94
Epoch 7, Loss: 20051.72
Epoch 8, Loss: 19520.00
Epoch 9, Loss: 18987.90
Epoch 10, Loss: 18570.72
Training BLSTM...
Epoch 1, Loss: 19301.37
Epoch 2, Loss: 3593.41
Epoch 3, Loss: 1336.27
Epoch 4, Loss: 671.00
Epoch 5, Loss: 382.65
Epoch 6, Loss: 217.78
Epoch 7, Loss: 151.65
Epoch 8, Loss: 101.28
Epoch 9, Loss: 73.52
Epoch 10, Loss: 52.80

Training Attention...
Epoch 1, Loss: 37606.86
Epoch 2, Loss: 30542.13
Epoch 3, Loss: 27779.07
Epoch 4, Loss: 26265.04
Epoch 5, Loss: 25080.90
Epoch 6, Loss: 24321.92
Epoch 7, Loss: 23529.78
Epoch 8, Loss: 22846.84
Epoch 9, Loss: 22256.49
Epoch 10, Loss: 21761.89
Epoch 11, Loss: 21296.13
Epoch 12, Loss: 21019.58
Epoch 13, Loss: 20607.86
Epoch 14, Loss: 20268.13
Epoch 15, Loss: 19993.47

=== VanillaRNN ===
Novelty: 1.0
Diversity: 1.0
Samples: ['kizaa mulrad nair', 'sheetal redwa', 'batesh k